# General canonical forms — interactive companion

Loads `general_canonical_forms.sage` (Brown–Dupont Proposition 6.7, arXiv:2501.03202) and `vertex_sum_canonical_forms.sage` (Proposition 6.10) into this notebook's kernel and shows how to call them directly, beyond what each script's own built-in test suite prints when run standalone.

**Requires the `sagemath` Jupyter kernel** (Kernel → Change Kernel, if this didn't open with it already). Must be opened from the same directory as the `.sage` files (or edit the `load(...)` paths below) — see `README.md` in this folder for how to sync this whole directory locally and launch Jupyter.

In [ ]:
load("general_canonical_forms.sage")

That single `load` also pulls in `common.sage` (vertex generators, `reduce_codim1`, etc. — see its own docstring) and runs `general_canonical_forms.sage`'s own test suite as a side effect, printing PASS/FAIL for the square-pyramid reproduction, the non-simple families, and the cross-check against Proposition 6.10. Scroll up to see that output; everything below is new exploration.

## Example 1: a family instance not already in the test suite

The cross-polytope at d=4 (16 facets, not simple) — pick any instance, any dimension.

In [ ]:
d = 4
y = [var(f"y{i}") for i in range(1, d + 1)]
P = Polyhedron(vertices=cross_polytope_vertices(d))
phi = general_canonical_form_density(P, y)
phi

In [ ]:
# The defining check: simple poles on exactly the polytope's own facets.
verify_pole_structure(f"Cross-polytope d={d}", phi, P, y)

## Example 2: bring your own polytope

Any vertex list works, as long as it's full-dimensional in the coordinates you pass (see `reduce_codim1` in `common.sage` if your polytope lives in a hyperplane of its natural ambient space, e.g. the permutohedron or associahedron). Here: a (non-simple, non-simplicial — genuinely general) square pyramid with a different apex height than the paper's own example, just to show it isn't hard-coded to that one case.

In [ ]:
pts = [(0, 0, 2), (1, 1, -1), (1, -1, -1), (-1, -1, -1), (-1, 1, -1)]
Q = Polyhedron(vertices=pts)
y3 = [var("y1"), var("y2"), var("y3")]
phi_Q = general_canonical_form_density(Q, y3)
print(phi_Q)
verify_pole_structure("Custom pyramid", phi_Q, Q, y3)

## Example 3: compare the general method (Prop. 6.7) against the simple-polytope shortcut (Prop. 6.10)

They must agree exactly on any simple polytope — this is one of the cross-checks `general_canonical_forms.sage` runs on load, repeated here on a fresh example (the associahedron, not in that script's own cross-check list).

In [ ]:
L = 4  # pentagon, d=2
y2 = [var("y1"), var("y2")]
pts = reduce_codim1(associahedron_vertices(L))
R = Polyhedron(vertices=pts)
phi_general = general_canonical_form_density(R, y2)
phi_simple = vertex_canonical_form_density(R, y2)
print("general:", phi_general)
print("simple: ", phi_simple)
print("agree exactly:", (phi_general - phi_simple).simplify_full() == 0)

## Example 4: inspect the nbc sets and flags at a single vertex

Useful for understanding *why* a term is (or isn't) present — this is what caught the square pyramid's apex having one nbc set (`{1,2,4}`) whose flag fails, matching the paper's own explanation.

In [ ]:
fdata = facet_data(Q)
for v in Q.vertex_generator():
    S_v = vertex_local_indices(v, fdata)
    print(v, "-> facets touching it:", [i + 1 for i in S_v])
    for I in nbc_bases(fdata, S_v):
        s = flag_sign(Q, I, fdata)
        print("   nbc", tuple(i + 1 for i in I), "-> flag sign:", s)